# 01 · Aşama 1 — TON_IoT veri hazırlama ve görülmemiş saldırı protokolü
**Amaç (Aşama 1):** normal trafik davranışını öğrenebilen ve daha önce görülmemiş saldırılar üzerinde
değerlendirilebilen bir veri yapısı oluşturmak.

Bu defter sırasıyla:
1. dosyayı yükler ve denetler (sınıf sayıları, eksik değerler, tekrarlar, etiket tutarlılığı);
2. kimlik / **veri sızıntısı** sütunlarını (`ts`, IP'ler, portlar) ve serbest metin alanlarını çıkarır;
3. **küresel tekrar temizliği** yapar (önce girdi sütunları, sonra nihai model öznitelik vektörü);
   aynı vektör hem normal hem saldırı etiketiyle görülürse **her etiketten bir kopya** tutulur ve işaretlenir;
4. rastgele bölme: normal 50/10/20/20 (eğitim/doğrulama/kalibrasyon/test); bilinen saldırılar 50/50
   (geliştirme-doğrulama / geliştirme-test); görülmeyecek aileler → yalnızca test;
5. kodlayıcı, öznitelik filtresi ve ölçekleyiciyi **yalnızca normal-eğitim** verisinde öğrenir;
6. tekrarlanabilirlik için her şeyi kaydeder (indeksler, tekrar sayıları, ön işlemci);
7. **IoT cihazında hesaplanabilen** öznitelik kümesini ayrıca hazırlar;
8. sızıntı kontrolü: aynı hattı kimlik sütunları *dahil* çalıştırır (yalnızca kontrol amaçlı).

In [ ]:
# ---- Kurulum (her defterde aynı): Drive'ı bağla, ortak kodu içe aktar ----
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_DIR = os.environ.get("TRUSTIDS_PROJECT", "/content/drive/MyDrive/TrustFed-TinyIDS-Final")
QUICK = os.environ.get("TRUSTIDS_QUICK", "0") == "1"   # hızlı mod YALNIZCA kodu test etmek içindir
# Devam modu: Colab oturumu kapanırsa, Drive'a kaydedilmiş tohumlar/adımlar atlanır ve kalan yerden devam edilir.
# Her şeyi baştan hesaplamak için: os.environ["TRUSTIDS_RESUME"] = "0"
RESUME = os.environ.get("TRUSTIDS_RESUME", "1") == "1"
sys.path.insert(0, PROJECT_DIR)

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from trustids.config import Config
from trustids import utils

# Veri kümesi seçimi: varsayılan TON_IoT; ikinci veri kümesi için TRUSTIDS_CONFIG=edge_iiotset.json
CONFIG_NAME = os.environ.get("TRUSTIDS_CONFIG", "default.json")
CFG_PATH = Path(PROJECT_DIR) / "configs" / CONFIG_NAME
if CONFIG_NAME != "default.json" and not CFG_PATH.exists():
    raise FileNotFoundError(f"{CFG_PATH} bulunamadı: configs/ klasörünü de Drive'a yükleyin.")
cfg = Config.load(CFG_PATH) if CFG_PATH.exists() else Config(project_dir=PROJECT_DIR)
cfg.project_dir = PROJECT_DIR
if QUICK:
    cfg.train_seeds, cfg.validity_resplits, cfg.epochs = cfg.train_seeds[:2], 30, 15
    cfg.random_control_repeats, cfg.bootstrap_repeats = 10, 200
    cfg.lofo_seeds, cfg.fl_orig_rounds = cfg.lofo_seeds[:2], 3
    cfg.resolution_cal_windows, cfg.resolution_eval_windows = 600, 600
cfg.check()
ART = cfg.artifacts
FIG = Path(PROJECT_DIR) / cfg.figures_dir; FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Proje klasörü:", PROJECT_DIR, "| hızlı mod:", QUICK, "| devam modu:", RESUME)
print("Drive'da kayıtlı ilerleme:", utils.progress(cfg))

In [ ]:
from trustids import data as D
raw = D.load_raw(cfg.data_path, cfg)
print(D.detect_version(raw))
audit = D.audit(raw)
pd.Series(audit["class_counts"]).rename("satır sayısı").to_frame()

In [ ]:
print("label/type uyuşmazlığı:", audit.get("label_type_mismatch"))
print("tüm sütunlarda birebir tekrar eden satır:", audit["exact_duplicate_rows"])
pd.Series(audit["dash_missing_per_column"]).rename("'-' değerleri").sort_values(ascending=False).to_frame().head(20)

## Hazırlama: tekrar temizliği, bölmeler, ön işleme (yalnızca normal-eğitim üzerinde)

In [ ]:
bundle = D.prepare(raw, cfg, include_leakage=False)
rep = bundle["report"]
print("Girdi sütunları:", len(rep["input_columns"]), "| kodlama/filtre sonrası model öznitelikleri:", rep["n_features"])
DEDUP = pd.DataFrame(rep["dedup"]["per_type"]).T.rename(columns={
    "before": "satır", "after_stage1": "kimliksiz_tekil_kayıt", "after": "tekil_model_vektörü"})
DEDUP["tekil_vektör_%"] = (100 * DEDUP["tekil_model_vektörü"] / DEDUP["satır"]).round(1)
print("Kimlik sütunları DAHİL birebir tekrar eden satır oranı: %.1f%%" % (100 * audit["exact_duplicate_rows"] / len(raw)))
DEDUP

### Tekrar temizliğinin üç düzeyi
1. **satır**: ham dosyadaki kayıt sayısı;
2. **kimliksiz tekil kayıt**: `ts`/IP/port gibi kimlik sütunları çıkarıldıktan sonra farklı kalan kayıtlar
   (bu kayıtlar ancak kimlik sütunlarıyla ayırt edilebiliyordu);
3. **tekil model vektörü**: modelin gördüğü girdi vektörü düzeyinde tekil örnekler (ör. `dst_ip_bytes` ≈ `dst_bytes`
   korelasyon filtresiyle çıkarıldığında yalnızca bu sütunda farklılaşan kayıtlar birleşir).

Dikkat: birçok satır **farklı akışlardır** ancak kimlik sütunları olmadan birbirinden ayırt edilemez. Model bu kayıtları
aynı örnek olarak görür; bu nedenle tüm sonuçlar tekil model vektörleri üzerinde raporlanır, satır ağırlıklı sonuçlar
ikincil olarak verilir.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.8))
x = np.arange(len(DEDUP))
ax.bar(x - 0.27, DEDUP["satır"], 0.27, label="satır")
ax.bar(x, DEDUP["kimliksiz_tekil_kayıt"], 0.27, label="kimliksiz tekil kayıt")
ax.bar(x + 0.27, DEDUP["tekil_model_vektörü"], 0.27, label="tekil model vektörü")
ax.set_xticks(x); ax.set_xticklabels(DEDUP.index, rotation=30, ha="right")
ax.set_yscale("log"); ax.set_ylabel("örnek sayısı (log)")
ax.set_title("TON_IoT: sınıf başına satır ve tekil örnek sayıları"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "01_dedup_before_after.png", dpi=150)

In [ ]:
print({k: v for k, v in rep["dedup"].items() if k != "per_type"})
print("Bölmeler arasında paylaşılan aynı vektörler. Beklenen: normal bölmeler arasında ~0; "
      "normal–saldırı örtüşmeleri işaretlenmiş çakışmalardır (her etiketten bir kopya tutulur):")
print(rep["residual_cross_split_identical_vectors"])
pd.DataFrame(rep["split_by_type"]).fillna(0).astype(int)

In [ ]:
n_cal = rep["split_sizes"]["benign_cal"]
rule = D.calibration_rule(n_cal, cfg.alpha, cfg.alpha_grid)
print("n_cal kuralı ((n_cal+1)·α ≥ 20):", rule)
mitm_n = int(np.sum(bundle["types"]["heldout_test"] == "mitm"))
rep["mitm_unique_vectors"] = mitm_n
rep["mitm_in_macro_average"] = mitm_n >= 200
print("mitm benzersiz vektör:", mitm_n, "-> makro ortalamaya dahil:", mitm_n >= 200)
rep["calibration_rule"] = rule
pd.Series(rep["dropped_columns"]).rename("çıkarılma nedeni").to_frame()

In [ ]:
out = ART / "prepared"
D.save_prepared(bundle, out)
utils.save_json(rep, out / "prepare_report.json")
utils.save_json({"features": bundle["preprocessor"].feature_names_,
                 "groups": bundle["preprocessor"].groups_}, out / "features.json")
print("Kaydedildi:", out)

## IoT cihazında hesaplanabilen öznitelik kümesi (Aşama 1 gereksinimi)
Yalnızca akış sayaçları (DNS/SSL/HTTP alanları derin paket incelemesi gerektirdiği için hariç). Aşama 2'de karşılaştırma olarak, Aşama 7'de (TinyML) aday girdi olarak kullanılır.

In [ ]:
dev = D.prepare(raw, cfg, feature_set="device")
D.save_prepared(dev, ART / "prepared_device")
utils.save_json(dev["report"], ART / "prepared_device" / "prepare_report.json")
print("Cihaz öznitelik kümesi:", dev["report"]["input_columns"], "->", dev["report"]["n_features"], "model özniteliği")

## Sızıntı kontrolü: aynı hat `ts`, IP ve port sütunları *dahil*
Yalnızca kimlik sütunlarının sonuçları şişirip şişirmediğini göstermek içindir; ana sonuçlarda **kullanılmaz**.

In [ ]:
RUN_LEAKAGE_CHECK = True
if RUN_LEAKAGE_CHECK:
    leak = D.prepare(raw, cfg, include_leakage=True)
    D.save_prepared(leak, ART / "prepared_leakage")
    utils.save_json(leak["report"], ART / "prepared_leakage" / "prepare_report.json")
    print("Sızıntı varyantı öznitelik sayısı:", leak["report"]["n_features"])

## Aşama 1 denetim özeti (profesöre gönderilecek)

In [ ]:
info = utils.load_json(ART / "data_file_info.json") if (ART / "data_file_info.json").exists() else D.detect_version(raw)
AUDIT = {
    "dosya": {k: info.get(k) for k in ["version", "rows", "columns", "has_ts", "sha256"]},
    "tekrar_temizligi": {k: v for k, v in rep["dedup"].items() if k != "per_type"},
    "sinif_basina_once_sonra": rep["dedup"]["per_type"],
    "bolme_boyutlari": rep["split_sizes"],
    "n_cal_kurali": rule,
    "mitm_benzersiz": mitm_n,
    "model_oznitelik_sayisi": rep["n_features"],
    "cihaz_oznitelikleri": dev["report"]["input_columns"],
    "cikarilan_sutunlar": rep["dropped_columns"],
}
utils.save_json(AUDIT, FIG / "01_faz1_denetim_ozeti.json")
pd.json_normalize(AUDIT["tekrar_temizligi"]).T.rename(columns={0: "değer"})

## Aşama 1 çıktıları
* `artifacts/prepared/` — öznitelik matrisleri, etiketler, bölme indeksleri, tekrar sayıları, ön işlemci
* `artifacts/prepared/prepare_report.json` — denetim, tekrar temizliği, bölme boyutları, çıkarılan sütunlar, n_cal kuralı
* `figures/01_dedup_before_after.png`, `figures/01_faz1_denetim_ozeti.json`

In [ ]:
# Colab: Drive'a yazılan tüm dosyaların kaydedildiğinden emin ol (oturumu kapatmadan önce mutlaka çalıştırın)
print("Kaydedilen ilerleme:", utils.progress(cfg))
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive kaydedildi ve ayrıldı. Sonraki defter Drive'ı yeniden bağlar.")